### 1. Imports + Setup

In [2]:
import pandas as pd
import numpy as np
import os
import glob

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

### 2. Load Data

In [4]:
# World Bank Data

wb_path = "../data/world_bank_data"

wb_files = [
    f
    for f in glob.glob(os.path.join(wb_path, "*.csv"))
    if not os.path.basename(f).startswith("._")
]

print("Files found:", len(wb_files))

wb_dfs = []
for file in wb_files:
    df = pd.read_csv(file)
    df["source_file"] = os.path.basename(file)
    wb_dfs.append(df)

print("DFs created:", len(wb_dfs))

wb_df = pd.concat(wb_dfs, ignore_index=True)
wb_df = wb_df.drop(columns=["obs_status", "unit"], errors="ignore")

print("World Bank shape:", wb_df.shape)
wb_df.head()

Files found: 264


KeyboardInterrupt: 

In [ ]:
# IMF Data

imf_path = "../data/International-Monetary-Fund/imf.csv"
imf_df = pd.read_csv(imf_path)
print("IMF shape:", imf_df.shape)
imf_df.head()

In [ ]:
# OECD Data

oecd_path = "../data/OECD/"
oecd_files = glob.glob(os.path.join(oecd_path, "*.csv"))

oecd_dfs = []
for file in oecd_files:
    df = pd.read_csv(file)
    df["source_file"] = os.path.basename(file)
    oecd_dfs.append(df)

oecd_df = pd.concat(oecd_dfs, ignore_index=True)
oecd_df = oecd_df.drop(columns=["obs_status", "unit"], errors="ignore")
print("OECD shape:", oecd_df.shape)
oecd_df.head()

### 3. Cleaning

In [ ]:
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df


wb_df = clean_columns(wb_df)
imf_df = clean_columns(imf_df)
oecd_df = clean_columns(oecd_df)

### 4.Null Analysis

In [ ]:
def null_summary(df, name):
    print(f"\n{name} NULL SUMMARY")
    nulls = df.isnull().sum()
    pct = (nulls / len(df)) * 100

    summary = pd.DataFrame({"null_count": nulls, "null_pct": pct}).sort_values(
        by="null_pct", ascending=False
    )

    return summary


wb_nulls = null_summary(wb_df, "World Bank")
imf_nulls = null_summary(imf_df, "IMF")
oecd_nulls = null_summary(oecd_df, "OECD")

wb_nulls.head(10)

### 5. Basic EDA

In [ ]:
print("WB:", wb_df.shape)
print("IMF:", imf_df.shape)
print("OECD:", oecd_df.shape)

In [ ]:
wb_df.dtypes

In [ ]:
wb_df.describe()

In [ ]:
wb_df["country"].nunique()
wb_df["year"].nunique()

In [ ]:
wb_df.columns.tolist()
wb_df["value"] = pd.to_numeric(wb_df["value"], errors="coerce")
wb_df["indicator"].drop_duplicates().sample(20)

In [ ]:
gdp_df = wb_df[wb_df["indicator"].str.contains("GDP", case=False, na=False)]
inflation_df = wb_df[wb_df["indicator"].str.contains("inflation", case=False, na=False)]
unemp_df = wb_df[wb_df["indicator"].str.contains("unemployment", case=False, na=False)]

gdp_df.head()

In [ ]:
gdp_df = wb_df[wb_df["indicator_id"] == "NY.GDP.MKTP.CD"].copy()
gdp_df["value"] = pd.to_numeric(gdp_df["value"], errors="coerce")

vals = gdp_df["value"].dropna().sample(1000, random_state=42)

plt.hist(np.log1p(vals), bins=30)
plt.title("Log GDP Distribution (Sampled Data)")
plt.xlabel("log(1 + GDP)")
plt.show()

In [ ]:
inflation_vals = inflation_df["value"].dropna()

# clip to reasonable range
inflation_vals_clipped = inflation_vals.clip(-20, 50)

plt.hist(inflation_vals_clipped.sample(1000, random_state=42), bins=30)
plt.title("Inflation Distribution (Clipped)")
plt.show()

In [ ]:
country_name = "United States"

us_macro = wb_df[
    (wb_df["country"] == country_name)
    & (
        wb_df["indicator_id"].isin(
            ["NY.GDP.MKTP.CD", "FP.CPI.TOTL.ZG", "SL.UEM.TOTL.ZS"]
        )
    )
].copy()

us_macro["value"] = pd.to_numeric(us_macro["value"], errors="coerce")

pivot_df = us_macro.pivot_table(index="year", columns="indicator_id", values="value")

pivot_df = pivot_df.rename(
    columns={
        "NY.GDP.MKTP.CD": "GDP_current_USD",
        "FP.CPI.TOTL.ZG": "inflation",
        "SL.UEM.TOTL.ZS": "unemployment",
    }
)

normalized = pivot_df / pivot_df.iloc[0]

normalized.plot(figsize=(10, 6))
plt.title("Normalized Macro Indicators (Indexed to First Year)")
plt.ylabel("Relative Change")
plt.show()

In [ ]:
### Save cleaned datasets

wb_df.to_csv("../data/clean_wb.csv", index=False)
imf_df.to_csv("../data/clean_imf.csv", index=False)
oecd_df.to_csv("../data/clean_oecd.csv", index=False)

print("Saved cleaned datasets!")